In [7]:
import subprocess, sys

packages = [
    'opencv-python',
    'mediapipe',
    'numpy',
    'scipy',
    'twilio',
    'pygame'
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('\n All libraries installed! Proceed to Cell 2.')

Installing opencv-python...
Installing mediapipe...
Installing numpy...
Installing scipy...
Installing twilio...
Installing pygame...

 All libraries installed! Proceed to Cell 2.


In [ ]:
# ─────────────────────────────────────────────────
#  FILL IN YOUR TWILIO DETAILS HERE
# ─────────────────────────────────────────────────

TWILIO_SID = "your_twilio_sid_here"
TWILIO_AUTH = "your_twilio_auth_token_here"
TWILIO_FROM = "your_twilio_number"
TWILIO_TO = "your_phone_number"                         # Driver's number (with country code)

# ─────────────────────────────────────────────────
# Alert cooldown — minimum seconds between two Twilio alerts
ALERT_COOLDOWN_SECONDS = 60
# ─────────────────────────────────────────────────

print(' Twilio credentials saved! Proceed to Cell 3.')

 Twilio credentials saved! Proceed to Cell 3.


In [9]:
# ── EYE settings ───────────────────────────────────
EAR_THRESHOLD      = 0.25   # Below this = eye closed
EAR_CONSEC_FRAMES  = 20     # Frames eye must stay closed before alert

# ── MOUTH / YAWN settings ──────────────────────────
MAR_THRESHOLD      = 0.65
MAR_CONSEC_FRAMES  = 15

# ── HEAD POSE settings ─────────────────────────────
HEAD_PITCH_THRESHOLD = 25   # Degrees downward nod before alert
HEAD_ROLL_THRESHOLD  = 30   # Degrees sideways tilt before alert  ← NEW
HEAD_CONSEC_FRAMES   = 15

# ── COLLAPSE detection ─────────────────────────────
COLLAPSE_SPEED_THRESHOLD = 18

# ── DISPLAY settings ───────────────────────────────
SHOW_LANDMARKS = True
CAMERA_INDEX   = 0

print(' Settings configured! Proceed to Cell 4.')

 Settings configured! Proceed to Cell 4.


In [10]:
import cv2
import numpy as np
import mediapipe as mp
from scipy.spatial import distance as dist

# ── MediaPipe setup (compatible with new mediapipe >= 0.10) ───────────────
try:
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing   = mp.solutions.drawing_utils
    mp_styles    = mp.solutions.drawing_styles
except AttributeError:
    # mediapipe >= 0.10.x fallback
    from mediapipe.python.solutions import face_mesh as mp_face_mesh
    from mediapipe.python.solutions import drawing_utils as mp_drawing
    from mediapipe.python.solutions import drawing_styles as mp_styles

# ── Landmark indices ──────────────────────────────────────────────────────
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33,  160, 158, 133, 153, 144]
MOUTH     = [61, 291, 39, 269, 0, 17]
POSE_IDS  = [1, 152, 263, 33, 287, 57]

MODEL_POINTS = np.array([
    (0.0,    0.0,    0.0),
    (0.0,   -330.0, -65.0),
    (-225.0,  170.0, -135.0),
    (225.0,   170.0, -135.0),
    (-150.0, -150.0, -125.0),
    (150.0,  -150.0, -125.0),
], dtype=np.float64)

# ── EAR ──────────────────────────────────────────────────────────────────
def eye_aspect_ratio(lms, indices, w, h):
    pts = np.array([(lms[i].x * w, lms[i].y * h) for i in indices])
    A = dist.euclidean(pts[1], pts[5])
    B = dist.euclidean(pts[2], pts[4])
    C = dist.euclidean(pts[0], pts[3])
    return (A + B) / (2.0 * C)

# ── MAR ──────────────────────────────────────────────────────────────────
def mouth_aspect_ratio(lms, indices, w, h):
    pts = np.array([(lms[i].x * w, lms[i].y * h) for i in indices])
    A = dist.euclidean(pts[2], pts[5])
    B = dist.euclidean(pts[3], pts[4])
    C = dist.euclidean(pts[0], pts[1])
    return (A + B) / (2.0 * C)

# ── Head Pose ─────────────────────────────────────────────────────────────
def get_head_pose(lms, w, h):
    image_pts = np.array([(lms[i].x * w, lms[i].y * h) for i in POSE_IDS], dtype=np.float64)
    focal     = w
    cam_mat   = np.array([[focal,0,w/2],[0,focal,h/2],[0,0,1]], dtype=np.float64)
    ok, rvec, tvec = cv2.solvePnP(MODEL_POINTS, image_pts, cam_mat,
                                   np.zeros((4,1)), flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok:
        return 0.0, 0.0, 0.0
    rmat, _ = cv2.Rodrigues(rvec)
    angles, *_ = cv2.RQDecomp3x3(rmat)
    return angles[0], angles[1], angles[2]  # pitch, yaw, roll

# ── Detector class ────────────────────────────────────────────────────────
class DrowsinessDetector:
    def __init__(self):
        self.face_mesh = mp_face_mesh.FaceMesh(
            max_num_faces=1, refine_landmarks=True,
            min_detection_confidence=0.6, min_tracking_confidence=0.6
        )
        self.eye_counter      = 0
        self.head_counter     = 0   # forward nod counter
        self.roll_counter     = 0   # sideways tilt counter
        self.yawn_counter     = 0
        self.collapse_counter = 0   # collapse frame counter (3 sec = ~90 frames)
        self.prev_nose_y      = None
        self.ear = self.mar = self.pitch = self.roll = 0.0

    def process(self, frame):
        h, w = frame.shape[:2]
        results = self.face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        alerts = dict(eye_closed=False, head_nod=False, head_tilt=False,
                      collapse=False, yawning=False)

        if not results.multi_face_landmarks:
            cv2.putText(frame, 'NO FACE DETECTED', (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
            self.prev_nose_y = None
            return frame, alerts

        fl  = results.multi_face_landmarks[0]
        lms = fl.landmark

        if SHOW_LANDMARKS:
            mp_drawing.draw_landmarks(
                frame, fl, mp_face_mesh.FACEMESH_CONTOURS,
                None, mp_styles.get_default_face_mesh_contours_style())

        # EAR
        self.ear = (eye_aspect_ratio(lms, LEFT_EYE, w, h) +
                    eye_aspect_ratio(lms, RIGHT_EYE, w, h)) / 2.0
        if self.ear < EAR_THRESHOLD:
            self.eye_counter += 1
            if self.eye_counter >= EAR_CONSEC_FRAMES:
                alerts['eye_closed'] = True
        else:
            self.eye_counter = 0

        # MAR
        self.mar = mouth_aspect_ratio(lms, MOUTH, w, h)
        if self.mar > MAR_THRESHOLD:
            self.yawn_counter += 1
            if self.yawn_counter >= MAR_CONSEC_FRAMES:
                alerts['yawning'] = True
        else:
            self.yawn_counter = 0

        # Head Pose — pitch (nod) + roll (sideways tilt)
        self.pitch, yaw, self.roll = get_head_pose(lms, w, h)

        # Forward nod detection
        if self.pitch < -HEAD_PITCH_THRESHOLD:
            self.head_counter += 1
            if self.head_counter >= HEAD_CONSEC_FRAMES:
                alerts['head_nod'] = True
        else:
            self.head_counter = 0

        # Sideways tilt detection (roll left or right)
        if abs(self.roll) > HEAD_ROLL_THRESHOLD:
            self.roll_counter += 1
            if self.roll_counter >= HEAD_CONSEC_FRAMES:
                alerts['head_tilt'] = True
        else:
            self.roll_counter = 0

        # Collapse — must sustain fast drop for ~3 seconds (90 frames at 30fps)
        nose_y = lms[1].y * h
        if self.prev_nose_y and (nose_y - self.prev_nose_y) > COLLAPSE_SPEED_THRESHOLD:
            self.collapse_counter += 1
            if self.collapse_counter >= 90:   # ~3 seconds at 30fps
                alerts['collapse'] = True
        else:
            self.collapse_counter = 0
        self.prev_nose_y = nose_y

        # HUD
        self._hud(frame, alerts, h, w)
        return frame, alerts

    def _hud(self, frame, alerts, h, w):
        ov = frame.copy()
        cv2.rectangle(ov, (0,0), (290,180), (0,0,0), -1)
        cv2.addWeighted(ov, 0.45, frame, 0.55, 0, frame)
        def put(t, y, c=(255,255,255)):
            cv2.putText(frame, t, (8,y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, c, 1)
        put(f'EAR  : {self.ear:.2f}',  25, (0,255,0) if self.ear >= EAR_THRESHOLD else (0,0,255))
        put(f'MAR  : {self.mar:.2f}',  50, (0,255,0) if self.mar <= MAR_THRESHOLD else (0,165,255))
        put(f'Pitch: {self.pitch:.1f}', 75, (0,255,0) if self.pitch >= -HEAD_PITCH_THRESHOLD else (0,0,255))
        put(f'Roll : {self.roll:.1f}', 100, (0,255,0) if abs(self.roll) <= HEAD_ROLL_THRESHOLD else (0,0,255))
        put(f'Eyes : {self.eye_counter}/{EAR_CONSEC_FRAMES}', 125)
        put(f'Head : {self.head_counter}/{HEAD_CONSEC_FRAMES}', 150)
        put(f'Tilt : {self.roll_counter}/{HEAD_CONSEC_FRAMES}', 175)

        if alerts['eye_closed']:
            cv2.rectangle(frame,(0,h-55),(w,h),(0,0,180),-1)
            cv2.putText(frame,'⚠ EYES CLOSED — DROWSY!',(w//2-195,h-15),
                        cv2.FONT_HERSHEY_DUPLEX,1,(255,255,255),2)
        if alerts['head_nod']:
            cv2.rectangle(frame,(0,h-110),(w,h-55),(0,100,200),-1)
            cv2.putText(frame,'⚠ HEAD NODDING!',(w//2-130,h-68),
                        cv2.FONT_HERSHEY_DUPLEX,1,(255,255,255),2)
        if alerts['head_tilt']:
            cv2.rectangle(frame,(0,h-165),(w,h-110),(0,140,255),-1)
            cv2.putText(frame,'⚠ HEAD TILTED SIDEWAYS!',(w//2-200,h-123),
                        cv2.FONT_HERSHEY_DUPLEX,1,(255,255,255),2)
        if alerts['collapse']:
            cv2.rectangle(frame,(0,0),(w,55),(0,0,255),-1)
            cv2.putText(frame,'SUDDEN COLLAPSE!',(w//2-170,38),
                        cv2.FONT_HERSHEY_DUPLEX,1,(255,255,0),2)
        if alerts['yawning']:
            cv2.putText(frame,'YAWNING',(w-130,30),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,165,255),2)

print(' Detector ready! Proceed to Cell 5.')

 Detector ready! Proceed to Cell 5.


In [ ]:
import time, threading

# ── Audio beep (Windows-compatible fix) ──────────────────────────────────
def play_beep(freq=1000, duration_ms=500, repeat=3):
    """Cross-platform beep: uses winsound on Windows, pygame on others."""
    import sys
    if sys.platform == 'win32':
        import winsound
        for _ in range(repeat):
            winsound.Beep(freq, duration_ms)
    else:
        try:
            import pygame
            pygame.mixer.init()
            import numpy as np
            sr = 44100
            t  = np.linspace(0, duration_ms/1000, int(sr*duration_ms/1000), False)
            w  = (np.sin(2*np.pi*freq*t)*32767).astype(np.int16)
            stereo = np.column_stack([w, w])
            s = pygame.sndarray.make_sound(stereo)
            for _ in range(repeat):
                s.play()
                pygame.time.wait(duration_ms)
        except Exception as e:
            print(f'⚠ Audio fallback (terminal bell): {e}')
            for _ in range(repeat):
                print('\a', end='', flush=True)

# ── Twilio ────────────────────────────────────────────────────────────────
try:
    from twilio.rest import Client
    twilio_client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
    TWILIO_OK = True
    print('Twilio connected!')
except Exception as e:
    TWILIO_OK = False
    print(f'Twilio not connected: {e}')

# ── Alert messages — separate for nod vs sideways tilt ───────────────────
CALL_MESSAGES = {
    'eye_closed': 'Warning! Eyes closed too long. Drowsiness detected. Pull over safely now.',
    'head_nod':   'Warning! Driver head is nodding forward. Severe drowsiness. Stop the vehicle safely.',
    'head_tilt':  'Warning! Driver head is tilted sideways. Possible loss of consciousness. Stop the vehicle now.',
    'collapse':   'Emergency! Sudden driver collapse detected. Immediate assistance required.',
    'yawning':    'Alert! Driver is yawning repeatedly. Please take a rest break soon.',
}

SMS_MESSAGES = {
    'eye_closed': ' DROWSY ALERT: Eyes closed too long! Pull over immediately!',
    'head_nod':   ' DROWSY ALERT: Head nodding forward! Stop safely!',
    'head_tilt':  ' TILT ALERT: Head tilted sideways! Driver may be unconscious! Stop now!',
    'collapse':   ' EMERGENCY: Driver collapse detected! Help needed now!',
    'yawning':    ' FATIGUE: Driver yawning repeatedly. Take a break!',
}

_last_alert = 0

def send_alerts(alert_type):
    global _last_alert
    # Beep always plays immediately for every alert type
    freq = {'collapse':1200, 'eye_closed':1000, 'head_nod':880,
            'head_tilt':950, 'yawning':660}.get(alert_type, 880)
    threading.Thread(target=play_beep, args=(freq, 500, 3), daemon=True).start()

    # Twilio with cooldown
    if time.time() - _last_alert < ALERT_COOLDOWN_SECONDS:
        remaining = int(ALERT_COOLDOWN_SECONDS - (time.time() - _last_alert))
        print(f'[TWILIO] Cooldown active — {remaining}s left')
        return
    _last_alert = time.time()

    def _call_and_sms():
        if not TWILIO_OK: return
        msg = CALL_MESSAGES.get(alert_type, 'Driver alert! Please check on the driver.')
        twiml = f'<Response><Say voice="alice" loop="3">{msg}</Say></Response>'
        try:
            c = twilio_client.calls.create(twiml=twiml,
                                            to=DRIVER_PHONE_NUMBER,
                                            from_=TWILIO_FROM_NUMBER)
            print(f'[TWILIO]  Call ({alert_type}): {c.sid}')
        except Exception as e:
            print(f'[TWILIO] Call failed: {e}')
        try:
            sms = SMS_MESSAGES.get(alert_type, ' Driver alert detected!')
            m = twilio_client.messages.create(body=sms,
                                               to=DRIVER_PHONE_NUMBER,
                                               from_=TWILIO_FROM_NUMBER)
            print(f'[TWILIO]  SMS ({alert_type}): {m.sid}')
        except Exception as e:
            print(f'[TWILIO] SMS failed: {e}')

    threading.Thread(target=_call_and_sms, daemon=True).start()

print(' Alert system ready! ')

Twilio connected!
 Alert system ready! Proceed to Cell 6.


In [12]:
import cv2, time

detector = DrowsinessDetector()
cap      = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f' Cannot open camera {CAMERA_INDEX}. Try changing CAMERA_INDEX to 1 in Cell 3.')
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    cap.set(cv2.CAP_PROP_FPS, 30)

    sent = dict(eye_closed=False, head_nod=False, head_tilt=False,
                collapse=False, yawning=False)
    fps_t, fps_c, fps = time.time(), 0, 0

    print(' Webcam window opened! Press Q to quit, R to reset, S to screenshot.')

    while True:
        ret, frame = cap.read()
        if not ret:
            print(' Camera read failed.'); break

        frame, alerts = detector.process(frame)

        # Trigger alerts
        for atype, triggered in alerts.items():
            if triggered and not sent[atype]:
                print(f' ALERT: {atype.upper()}')
                send_alerts(atype)
                sent[atype] = True
            elif not triggered:
                sent[atype] = False

        # FPS
        fps_c += 1
        if time.time() - fps_t >= 1.0:
            fps = fps_c; fps_c = 0; fps_t = time.time()
        cv2.putText(frame, f'FPS:{fps}', (frame.shape[1]-80, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180,180,180), 1)

        # Status bar
        active = [k.upper() for k,v in alerts.items() if v]
        status = '  |  '.join(active) if active else 'NORMAL'
        color  = (0,255,0) if status == 'NORMAL' else (0,0,255)
        cv2.putText(frame, f'STATUS: {status}',
                    (10, frame.shape[0]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow(' Driver Drowsiness Detection  [Q=Quit  R=Reset  S=Screenshot]', frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print(' Quit. Stopping...'); break
        elif key == ord('r'):
            detector.eye_counter = detector.head_counter = detector.yawn_counter = 0
            detector.roll_counter = detector.collapse_counter = 0
            sent = {k:False for k in sent}
            print(' Counters reset.')
        elif key == ord('s'):
            import os; os.makedirs('screenshots', exist_ok=True)
            fn = f'screenshots/screenshot_{int(time.time())}.jpg'
            cv2.imwrite(fn, frame)
            print(f' Screenshot saved: {fn}')

    cap.release()
    cv2.destroyAllWindows()
    print(' Detection stopped. Camera released.')

 Webcam window opened! Press Q to quit, R to reset, S to screenshot.
 ALERT: EYE_CLOSED
 ALERT: HEAD_TILT
[TWILIO] Cooldown active — 58s left
[TWILIO]  Call (eye_closed): CA18b28f53bec575b8e6bf382d9b8375ba
[TWILIO]  SMS (eye_closed): SM9f566fafd9b4ede01c4d10565633efb3
 Quit. Stopping...
 Detection stopped. Camera released.
